# PointLeg Colab 启动入口

按顺序运行本 Notebook：克隆项目、安装锁定依赖、确认 GPU 和任务注册，再按需启动训练。训练默认关闭，先确认任务与环境数量。

In [ ]:
import sys

if sys.version_info < (3, 10):
    raise RuntimeError(f'PointLeg requires Python 3.10 or newer, got {sys.version}')

print(f'Python: {sys.version.split()[0]}')

In [ ]:
import os
import subprocess
from pathlib import Path

REPO_URL = 'https://github.com/aoioi222/RL_pointleg.git'
PROJECT_DIR = Path('/content/RL_pointleg')

if PROJECT_DIR.exists():
    subprocess.run(['git', '-C', str(PROJECT_DIR), 'pull', '--ff-only'], check=True)
else:
    subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, str(PROJECT_DIR)], check=True)

os.chdir(PROJECT_DIR)
print(f'Working directory: {Path.cwd()}')

In [ ]:
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'uv'], check=True)
subprocess.run(['uv', 'sync', '--locked', '--extra', 'cu128'], check=True)

In [ ]:
subprocess.run(['nvidia-smi'], check=False)
subprocess.run(['uv', 'run', '--locked', '--no-sync', 'pointleg', 'list'], check=True)

## 训练配置

默认使用平地任务和较小并行环境数，便于先确认 Colab 的依赖与显存配置。模型权重未随仓库上传；如需续训或回放，请先将检查点上传至 Google Drive，并在下一单元中补充相应的 `--agent.resume`、`--agent.load-run` 和 `--agent.load-checkpoint` 参数。

In [ ]:
from google.colab import drive

MOUNT_GOOGLE_DRIVE = True
TASK_ID = 'PointLeg-Flat-v0'
NUM_ENVS = 64
MAX_ITERATIONS = 1000
RUN_NAME = 'colab_flat'

LOG_ROOT = PROJECT_DIR / 'logs'
if MOUNT_GOOGLE_DRIVE:
    drive.mount('/content/drive', force_remount=False)
    LOG_ROOT = Path('/content/drive/MyDrive/pointleg_logs')

LOG_ROOT.mkdir(parents=True, exist_ok=True)
print(f'Logs will be saved to: {LOG_ROOT}')

In [ ]:
START_TRAINING = False

train_command = [
    'uv', 'run', '--locked', '--no-sync', 'pointleg', 'train', TASK_ID,
    '--env.scene.num-envs', str(NUM_ENVS),
    '--agent.max-iterations', str(MAX_ITERATIONS),
    '--agent.run-name', RUN_NAME,
    '--agent.logger', 'tensorboard',
    '--agent.upload-model', 'False',
    '--log-root', str(LOG_ROOT),
]

print(' '.join(train_command))
if START_TRAINING:
    subprocess.run(train_command, check=True)
else:
    print('Training is disabled. Set START_TRAINING = True after confirming the command above.')